# Synthetic MLP Direct Prototype

A component playground for the direct old-style synthetic runner.

## 0. Knobs

In [ ]:
RUN_FULL_RUN = True

KNOBS = {
    "device": "cuda",
    "out_dir": "outputs/prototype_runners/synthetic_mlp_notebook",
    "run_name": "synthetic_notebook_check",
    "seed": 0,
    "data_dim": 2,
    "intrinsic_dim": 2,
    "n_train": 10,
    "n_reference": 1280,
    "n_clusters": 3,
    "sigma_z": 0.10,
    "sigma_eta": 1.0e-3,
    "geometry_seed": 0,
    "hidden_dim": 128,
    "n_layers": 3,
    "time_emb_dim": 32,
    "prediction_space": "v",
    "loss_space": "v",
    "n_steps": 500000,
    "batch_cap": 0,
    "lr": 1.0e-3,
    "weight_decay": 0.0,
    "log_every": 10,
    "metric_every": 10,
    "ckpt_every": 250,
    "eval_samples": 64,
    "sampling_method": "heun",
    "sampling_steps": 32,
    "ratio_threshold": 0.3,
    "chunk_size": 256,
    "sliced_w2_projections": 64,
    "wandb": False,
    "wandb_mode": "offline",
    "wandb_project": "mem_par",
    "wandb_entity": "",
    "wandb_name": "",
    "wandb_group": "",
    "wandb_tags": "",
    "wandb_notes": "",
    "wandb_strict": False,
}

## 1. Imports

In [ ]:
import json
import sys
from pathlib import Path

import torch
from IPython.display import Image, display

NOTEBOOK_DIR = Path.cwd()
for candidate in (NOTEBOOK_DIR, *NOTEBOOK_DIR.parents):
    if (candidate / "src").is_dir() and (candidate / "prototyping").is_dir():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find repo root")

for path in (REPO_ROOT, REPO_ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from gdfm import GDFM
from mp_config import config_to_dict, get_base_config
from mp_data import build_data_bundle
from prototyping.metrics_synthetic_controlled import (
    memorization_metrics_in_spaces,
    oracle_gmm_metrics,
    project_to_subspace,
)
from prototyping.runners import synthetic_mlp as runner

In [ ]:
    argv = [
        "--device", str(knobs["device"]),
        "--out-dir", str(knobs["out_dir"]),
        "--run-name", str(knobs["run_name"]),
        "--seed", str(knobs["seed"]),
        "--data-dim", str(knobs["data_dim"]),
        "--intrinsic-dim", str(knobs["intrinsic_dim"]),
        "--n-train", str(knobs["n_train"]),
        "--n-reference", str(knobs["n_reference"]),
        "--n-clusters", str(knobs["n_clusters"]),
        "--sigma-z", str(knobs["sigma_z"]),
        "--sigma-eta", str(knobs["sigma_eta"]),
        "--geometry-seed", str(knobs["geometry_seed"]),
        "--hidden-dim", str(knobs["hidden_dim"]),
        "--n-layers", str(knobs["n_layers"]),
        "--time-emb-dim", str(knobs["time_emb_dim"]),
        "--prediction-space", str(knobs["prediction_space"]),
        "--loss-space", str(knobs["loss_space"]),
        "--n-steps", str(knobs["n_steps"]),
        "--batch-cap", str(knobs["batch_cap"]),
        "--lr", str(knobs["lr"]),
        "--weight-decay", str(knobs["weight_decay"]),
        "--log-every", str(knobs["log_every"]),
        "--metric-every", str(knobs["metric_every"]),
        "--ckpt-every", str(knobs["ckpt_every"]),
        "--eval-samples", str(knobs["eval_samples"]),
        "--sampling-method", str(knobs["sampling_method"]),
        "--sampling-steps", str(knobs["sampling_steps"]),
        "--ratio-threshold", str(knobs["ratio_threshold"]),
        "--chunk-size", str(knobs["chunk_size"]),
        "--sliced-w2-projections", str(knobs["sliced_w2_projections"]),
        "--wandb-mode", str(knobs["wandb_mode"]),
        "--wandb-project", str(knobs["wandb_project"]),
    ]
    if knobs["wandb"]:
        argv.append("--wandb")
    if knobs["wandb_entity"]:
        argv.extend(["--wandb-entity", str(knobs["wandb_entity"])])
    if knobs["wandb_name"]:
        argv.extend(["--wandb-name", str(knobs["wandb_name"])])
    if knobs["wandb_group"]:
        argv.extend(["--wandb-group", str(knobs["wandb_group"])])
    if knobs["wandb_tags"]:
        argv.extend(["--wandb-tags", str(knobs["wandb_tags"])])
    if knobs["wandb_notes"]:
        argv.extend(["--wandb-notes", str(knobs["wandb_notes"])])
    if knobs["wandb_strict"]:
        argv.append("--wandb-strict")
    return argv

args = runner.parse_args(argv_from_knobs(KNOBS))
out_root = Path(args.out_dir).expanduser().resolve()
latest = out_root / "latest"
latest_checkpoints = out_root / "latest_checkpoints"
{"out_root": str(out_root), "latest": str(latest), "latest_checkpoints": str(latest_checkpoints)}

## 2. Config

In [ ]:
batching = {"batch_cap": args.batch_cap, "effective_batch_size": min(args.n_train, args.batch_cap), "drop_last": args.n_train > args.batch_cap}
effective_batch_size = min(args.n_train, args.batch_cap)
config = get_base_config()
config.study.name = runner.RUNNER_NAME
config.study.description = "Direct synthetic-data GDFM/MLP prototype runner."
config.outputs.run_name = args.run_name
config.framework.family = "fm"
config.framework.prediction_space = args.prediction_space
config.framework.loss_space = args.loss_space
config.data.dataset = "gmm_subspace"
config.data.root = "synthetic"
config.data.download = False
config.data.data_source = "canonical"
config.data.n_train = args.n_train
config.data.n_eval = args.n_reference
config.data.data_dim = args.data_dim
config.data.intrinsic_dim = args.intrinsic_dim
config.data.n_clusters = args.n_clusters
config.data.sigma_z = args.sigma_z
config.data.sigma_eta = args.sigma_eta
config.data.geometry_seed = args.geometry_seed
config.data.flatten = False
config.network.arch = "mlp"
config.network.hidden_dim = args.hidden_dim
config.network.n_layers = args.n_layers
config.network.time_emb_dim = args.time_emb_dim
config.training.seed = args.seed
config.training.n_steps = args.n_steps
config.training.batch_cap = int(args.batch_cap)
config.training.batch_size = int(args.batch_cap)
config.training.lr = args.lr
config.training.weight_decay = args.weight_decay
config.training.ckpt_every = args.ckpt_every
config.training.log_every = args.log_every
config.training.optimizer = "adam"
config.evaluation.n_samples = args.eval_samples
config.evaluation.ratio_threshold = args.ratio_threshold
config.evaluation.chunk_size = args.chunk_size
config.evaluation.sample_panel_count = min(8, args.eval_samples)
config.sampling.method = args.sampling_method
config.sampling.n_steps = args.sampling_steps
config.sliced_w2_projections = args.sliced_w2_projections
config_payload = config_to_dict(config)
{
    "data": config_payload["data"],
    "network": config_payload["network"],
    "framework": config_payload["framework"],
    "training": config_payload["training"],
}

## 3. Data

In [ ]:
data_bundle = build_data_bundle(config)

train_tensor = data_bundle["train"].contiguous()
reference_tensor = data_bundle["reference"].contiguous()
train_labels = data_bundle["train_labels"]
reference_labels = data_bundle["reference_labels"]
basis = data_bundle["geometry"]["basis"].contiguous()
latent_means = data_bundle["geometry"]["means"].contiguous()
ambient_means = latent_means @ basis.T
{
    "train_shape": list(train_tensor.shape),
    "reference_shape": list(reference_tensor.shape),
    "basis_shape": list(basis.shape),
    
}

## 4. Projection And Baselines

In [ ]:
train_parts = project_to_subspace(train_tensor, basis)
reference_parts = project_to_subspace(reference_tensor, basis)
train_latent = train_parts["latent"]
reference_latent = reference_parts["latent"]

z_eval = torch.randn(args.eval_samples, args.data_dim, generator=torch.Generator().manual_seed(args.seed))
sliced_generator = torch.Generator().manual_seed(args.seed + 1009)
sliced_directions = torch.randn(args.sliced_w2_projections, args.intrinsic_dim, generator=sliced_generator)
sliced_directions = sliced_directions / sliced_directions.norm(dim=1, keepdim=True).clamp(min=1.0e-12)

baseline_count = min(args.eval_samples, reference_tensor.shape[0])
baseline_tensor = reference_tensor[:baseline_count]
baseline_latent = reference_latent[:baseline_count]
baseline_oracle_metrics = oracle_gmm_metrics(
    baseline_tensor, basis, latent_means, args.sigma_z, args.sigma_eta, baseline_latent, sliced_directions
)
baseline_mem = memorization_metrics_in_spaces(
    baseline_tensor, train_tensor, basis, train_latent, chunk_size=args.chunk_size, config=config
)
{"baseline_mem": baseline_mem, "baseline_oracle": baseline_oracle_metrics}

## 5. Model And One Update

In [ ]:
device = args.device
if device == "auto":
    device = "cuda" if torch.cuda.is_available() else "cpu"
model = GDFM(config).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)

batch = train_tensor[:effective_batch_size].to(device)
optimizer.zero_grad(set_to_none=True)
loss = model.compute_loss(batch)
loss.backward()
grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0e9)
optimizer.step()
{
    "device": device,
    "batch_shape": list(batch.shape),
    "loss": float(loss.detach().cpu().item()),
    "grad_norm": float(grad_norm.detach().cpu().item()),
    "parameters": runner._count_parameters(model),
}

## 6. Metrics

In [ ]:
model.eval()
with torch.no_grad():
    samples = model.sample(
        z_eval.clone().to(device),
        n_steps=args.sampling_steps,
        method=args.sampling_method,
        device=device,
    ).cpu()
metric_record = runner._metric_record(
    step=0,
    samples=samples,
    train_tensor=train_tensor,
    train_latent=train_latent,
    basis=basis,
    latent_means=latent_means,
    baseline_mem=baseline_mem,
    baseline_oracle_metrics=baseline_oracle_metrics,
    baseline_latent=baseline_latent,
    sliced_directions=sliced_directions,
    args=args,
    config=config,
)
metric_record

## 7. Optional Full Run

In [ ]:
run_summary = None
if RUN_FULL_RUN:
    run_summary = runner.main(argv_from_knobs(KNOBS))
run_summary

## 8. Artifacts And Figures

In [ ]:
def load_latest_summary(root):
    latest_path = Path(root) / "latest"
    if latest_path.exists():
        summary_path = latest_path / "artifacts" / "run_summary.json"
        if summary_path.exists():
            return json.loads(summary_path.read_text()), latest_path
    return None, latest_path

summary, attempt_dir = (run_summary, Path(run_summary["run_dir"])) if run_summary else load_latest_summary(out_root)
if summary:
    print(json.dumps(summary.get("paths", {}), indent=2))
    for name, figure_path in summary.get("figures", {}).items():
        figure_path = Path(figure_path)
        if figure_path.suffix.lower() == ".png" and figure_path.exists():
            print(name, figure_path)
            display(Image(filename=str(figure_path)))
else:
    print(f"No completed synthetic run found at {attempt_dir}")